# Animal Hibernation Patterns: Data Analysis & Bayesian Modeling

## Overview
This notebook explores a biological dataset of animal hibernation patterns (`hibernation.csv`), covering body mass changes, hibernation duration, diet groups, and reproductive activity.

### Workflow
1. **Data Loading & Preprocessing:** Parsing timestamp columns and boolean flags with `pandas`.
2. **Exploratory Statistics:** Vectorized aggregation of sex distribution across diet types.
3. **Time-Series Utility:** Implementing a typed, doctested moving average function.
4. **Feature Engineering:** Calculating daily body mass loss across diets.
5. **Data Visualization:** Scatter plots and longitudinal per-animal hibernation trends using `matplotlib`.
6. **Bayesian Linear Regression:** Modeling post-hibernation body mass as a function of hibernation duration in reproductively active females using `PyMC` and `ArviZ`.

In [ ]:
import doctest
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az

## 1. Data Loading & Preprocessing
Load `hibernation.csv`, convert `hibstart`, `hibend`, and `hibendyearbefore` to `datetime` objects, and cast `repro_active` to boolean.

In [ ]:
# Load the dataset
hib = pd.read_csv('hibernation.csv')

# Convert date columns to proper timestamps
date_cols = ['hibstart', 'hibend', 'hibendyearbefore']
for col in date_cols:
    hib[col] = pd.to_datetime(hib[col], dayfirst=True)

# Convert reproductive activity flag to boolean
hib['repro_active'] = hib['repro_active'] == 'yes'

print("Data loaded. 'repro_active' dtype:", hib['repro_active'].dtype)
hib.head()

## 2. Exploratory Statistics: Sex Distribution Across Diet Groups
Compute total observations by sex and the relative percentage of males vs. females within each diet category without explicit loops.

In [ ]:
# Total count for each sex
sex_counts = hib['sex'].value_counts()
print("Observations per sex:\n", sex_counts)

# Percentage of males vs. females for each diet type
diet_sex_stats = hib.groupby('diet')['sex'].value_counts(normalize=True).unstack() * 100
print("\nPercentage of males vs. females per diet:")
print(diet_sex_stats)

## 3. Moving Average Function (Type Hints & Doctest)
Define a forward-looking moving average function with window size $w$ (returning a series $w - 1$ elements shorter than the input), verified with `doctest`.

In [ ]:
def moving_average(series: pd.Series, w: int) -> pd.Series:
    """
    Calculate the moving average of each value and the next (w - 1) subsequent values.
    Returns a Series with (w - 1) fewer elements than the input.

    >>> s1 = pd.Series([10.0, 20.0, 30.0, 40.0, 50.0])
    >>> list(moving_average(s1, 3))
    [20.0, 30.0, 40.0]

    >>> s2 = pd.Series([2.0, 4.0, 6.0, 8.0])
    >>> list(moving_average(s2, 2))
    [3.0, 5.0, 7.0]
    """
    result = series.rolling(window=w).mean().shift(-(w - 1))
    return result.dropna()

doctest.testmod()

## 4. Applying Moving Average to Sorted Body Mass Data
Apply `moving_average` (window size $w = 4$) to ascending sorted values of `bm_before` and `bm_after`.

In [ ]:
bm_before_sorted = hib['bm_before'].sort_values()
bm_before_ma = moving_average(bm_before_sorted, 4)

bm_after_sorted = hib['bm_after'].sort_values()
bm_after_ma = moving_average(bm_after_sorted, 4)

print("Moving average for Body Mass Before (first 5 values):")
print(bm_before_ma.head())

print("\nMoving average for Body Mass After (first 5 values):")
print(bm_after_ma.head())

## 5. Feature Engineering: Daily Mass Loss by Diet
Compute `mass_loss_per_day = (bm_before - bm_after) / hibdur days` and aggregate its mean per diet group.

In [ ]:
hib['mass_loss_per_day'] = (hib['bm_before'] - hib['bm_after']) / hib['hibdur days']
diet_mass_loss = hib.groupby('diet')['mass_loss_per_day'].mean()

print("Mean mass loss per day for each diet group:")
print(diet_mass_loss)

## 6. Visualization: Hibernation Duration vs. Age at Death
Scatter plot comparing hibernation duration and lifespan, grouped by sex.

In [ ]:
plt.figure(figsize=(9, 6))

males = hib[hib['sex'] == 'm']
females = hib[hib['sex'] == 'f']

plt.scatter(males['hibdur days'], males['age death'], color='darkblue', label='Males', alpha=0.6, marker='*')
plt.scatter(females['hibdur days'], females['age death'], color='red', label='Females', alpha=0.6, marker='*')
plt.grid(True, linestyle='--', alpha=0.5)

plt.xlabel('Hibernation Duration (days)')
plt.ylabel('Age at Death')
plt.title('Hibernation Duration vs. Age at Death by Sex')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Longitudinal Analysis: Individual Hibernation Trends Over Time
Track hibernation duration across observation years for each individual animal.

In [ ]:
plt.figure(figsize=(11, 7))

for animal_name, group in hib.groupby('a name'):
    group_sorted = group.sort_values('year')
    plt.plot(group_sorted['year'], group_sorted['hibdur days'], marker='o', label=animal_name)

plt.xlabel('Year of Observation')
plt.ylabel('Hibernation Duration (days)')
plt.title('Hibernation Length Trends for Each Animal')
plt.legend(title='Animal Name', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 8. Bayesian Linear Regression with PyMC & ArviZ
Focusing on reproductively active females, we model post-hibernation body mass (`bm_after`) as a linear function of hibernation duration ($D$):

$$
\begin{aligned}
\alpha &\sim \mathcal{N}(150, 50^2) \\
\beta &\sim \mathcal{N}(0, 10^2) \\
\sigma &\sim \mathrm{Exponential}(\lambda = 0.1) \\
Y_i &\sim \mathcal{N}(\alpha + \beta D_i, \sigma^2)
\end{aligned}
$$

We sample the posterior distributions using MCMC in `PyMC` and visualize credible intervals with `ArviZ`.

In [ ]:
# Filter for reproductively active females
model_data = hib[(hib['sex'] == 'f') & (hib['repro_active'] == True)]

D = model_data['hibdur days'].values
Y = model_data['bm_after'].values

# Define the Bayesian linear regression model
with pm.Model() as hibernation_model:
    alpha = pm.Normal('alpha', mu=150, sigma=50)
    beta = pm.Normal('beta', mu=0, sigma=10)
    sigma = pm.Exponential('sigma', lam=0.1)

    mu = alpha + beta * D
    obs = pm.Normal('obs', mu=mu, sigma=sigma, observed=Y)

    # Sample posterior distributions
    trace = pm.sample(1000, return_inferencedata=True)

az.plot_posterior(trace)
plt.show()